# 5. 深度学习计算（上）

## 5.1 层和块

> 之前的代码中，我们仅使用过 `nn.Sequential` 这个线性容器 (无法处理条件分支逻辑)
>
> 为了实现更复杂的架构，我们需要学习 PyTorch 的核心基类：<b>nn.Module<b>。

### 1. 核心概念：块 (Blocks)

在 PyTorch 中，**“块”**（Block）可以描述单个层、由多个层组成的组件，甚至整个模型本身。
*   **数学本质**：块是一个将输入张量转化为输出张量的函数。
*   **工程本质**：块是一个继承自 `nn.Module` 的类，它维护着**状态**（参数）和**行为**（前向传播）。

---


### 2. 自定义多层感知机块

In [6]:
import torch
from torch import nn, Tensor

class MLP(nn.Module):
    """自定义多层感知机块。
    
    展示了 nn.Module 的基本结构：__init__ 负责定义组件，forward 负责定义计算流。
    """
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        """初始化模型层。
        
        Args:
            input_dim: 输入特征维度。
            hidden_dim: 隐藏层神经元数量。
            output_dim: 输出类别数量。
        """
        # 必须调用父类的 __init__(), 否则无法自动注册参数
        super().__init__()

        # 将层定义为类的成员属性
        self.hidden = nn.Linear(input_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, X: Tensor) -> Tensor:
        """定义前向传播逻辑。
        
        Args:
            X: 输入向量。

        Returns:
            Tensor: 输出向量。
        """
        # 我们可以自由控制计算顺序，甚至可以在这里加入 Python 的 if/for 逻辑
        h = self.relu(self.hidden(X))
        return self.out(h)
    
# --- 使用示例 ---
net = MLP(input_dim=20, hidden_dim=256, output_dim=10)
X = torch.randn(2, 20)
# nn.Module 的魔法方法 __call__ 允许调用 net(X) 时执行 forward 逻辑
# 建议：始终使用 net(X) 而不是 net.forward(X), 因为直接调用 forward 可能会导致某些监控或调试工具失效。
output = net(X) 
print(f"自定义的 MLP 输出形状: {output.shape}")

自定义的 MLP 输出形状: torch.Size([2, 10])


---

### 3. 细致梳理：`nn.Module` 

为什么继承 `nn.Module` 后，模型就能自动求导并管理参数了？

1.  **参数注册 (Parameter Registration)**：
    当你执行 `self.hidden = nn.Linear(...)` 时，`nn.Module` 会在后台自动识别出这是一个包含参数的层，并将其权重和偏置加入到模型的 `parameters()` 列表中。
2.  **`__call__` 与 `forward`**：
    在 PyTorch 中，你调用 `net(X)` 而不是 `net.forward(X)`。这是因为父类实现了 `__call__` 方法，它在运行 `forward` 之前和之后会执行一些必要的“钩子”操作（如注册梯度）。
3.  **递归性**：
    一个 `nn.Module` 可以包含另一个 `nn.Module`。这允许我们像搭乐高一样，把小组件拼成大系统。

---


### 4. 实现一个自定义的顺序块 (Sequential)

In [7]:
class MySequential(nn.Module):
    """手动实现 Sequential 容器。"""
    def __init__(self, *args: nn.Module):
        super().__init__()
        # _modules 是父类 nn.Module 内部维护的一个有序字典
        for idx, module in enumerate(args):
            self._modules[str(idx)] = module

    def forward(self, X: Tensor) -> Tensor:
        # 按顺序执行每一个已注册的块
        for block in self._modules.values():
            X = block(X)
        return X
    
# --- 使用方法 ---
net_test = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
print(f"MySequential 输出形状: {net_test(torch.randn(2, 20)).shape}")

MySequential 输出形状: torch.Size([2, 10])


---

### 5. 在前向传播中执行复杂计算 (Fixed Parameters)

> 这是 nn.Sequential 无法做到的事：在计算过程中加入不参与训练的常数或复杂的控制流。


In [8]:
class FixedMLP(nn.Module):
    """展示在 forward 中加入复杂计算逻辑。"""
    def __init__(self):
        super().__init__()
        # rand_weight 是随机生成的，且不设为 Parameter, 因此不会被梯度更新
        self.rand_weight = torch.randn(20, 20)
        self.linear = nn.Linear(20, 20)

    def forward(self, X: Tensor) -> Tensor:
        X = self.linear(X)
        # 使用常数权重进行计算，并加入复杂的 Python 控制流
        X = torch.relu(X @ self.rand_weight + 1)

        # 复用同一个线性层 (参数共享)
        X = self.linear(X)

        # 控制流演示
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

# --- 使用方法 ---
net_current = FixedMLP()
X = torch.randn(2, 20)
output = net_current(X)

print(f"1. 输出结果: {output.item():.4f} (是一个标量)")

# 演示 1：证明 rand_weight 不在参数列表中（不会被训练）
params = [name for name, _ in net_current.named_parameters()]
print(f"2. 模型参数列表: {params}")
print(f"   注意：'rand_weight' 不在其中，因为它只是一个普通的 Tensor 而非 nn.Parameter")

# 演示 2：执行反向传播，查看梯度
output.backward()
print(f"3. linear.weight 的梯度形状: {net_current.linear.weight.grad.shape}")

# 演示 3：由于 forward 中有 while 循环，我们可以检查输出是否符合预期（sum < 1）
print(f"4. 最终 X.sum() 是否小于 1: {output < 1}")

1. 输出结果: -0.0828 (是一个标量)
2. 模型参数列表: ['linear.weight', 'linear.bias']
   注意：'rand_weight' 不在其中，因为它只是一个普通的 Tensor 而非 nn.Parameter
3. linear.weight 的梯度形状: torch.Size([20, 20])
4. 最终 X.sum() 是否小于 1: True


In [9]:
# 嵌套块练习
class NestMLP(nn.Module):
    """嵌套块练习：级联(串联)两个 MLP。"""
    def __init__(self):
        super().__init__()
        # 在容器中嵌套我们自定义的 MLP 块
        self.net = nn.Sequential(
            MLP(20, 64, 32),
            MLP(32, 16, 10)
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.net(X)
    
# --- 测试 ---
nest_net = NestMLP()
print(f"NestMLP 输出形状: {nest_net(torch.randn(2, 20)).shape}")

NestMLP 输出形状: torch.Size([2, 10])


### 6. 关于“块”的工程规范建议

**`nn.Module`** 的开发准则：

1.  **参数注册**：永远在 `__init__` 中将子模块赋值给 `self.xxx`，这样 `net.parameters()` 才能找到它们。
2.  **逻辑位置**：所有的参数初始化逻辑放在 `__init__`，所有的计算逻辑放在 `forward`。**严禁**在 `forward` 中定义新的 `nn.Linear`（否则每次运行都会创建新权重）。
3.  **类型注解**：在 `forward` 中明确标注 `Tensor` 的输入输出，这对于大型模型 Debug 至关重要。

---


## 5.2 参数管理

### 1. 参数访问

当我们使用 nn.Sequential 或自定义 nn.Module 组建网络后，参数被组织在一个嵌套的层次结构中。

#### 1.1 访问特定层的参数

我们可以通过索引或属性名来获取特定层的 state_dict（状态字典）。

In [10]:
import torch
from torch import nn, Tensor

def inspect_parameters(net: nn.Module) -> None:
    """演示如何访问模型不同层的参数。"""

    # 1. 访问特定层的参数
    # net[2] 表示访问 Sequential 中的第三层 (0-based)
    print("Layer 2 State Dict Keys:", net[2].state_dict().keys())

    # 2. 访问具体的权重对象
    # weight 是一个 nn.Parameter 对象
    weight_param = net[2].weight
    print(f"权重类型: {type(weight_param)}")
    print(f"权重数值 (Tensor):\n{weight_param.data}")
    print(f"权重梯度: {weight_param.grad}") # 训练前为 None

# --- 示例 ---
net = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 1))
X = torch.randn(2, 4)
net(X) # 运行一次前向传播以确保参数被初始化

inspect_parameters(net)

Layer 2 State Dict Keys: odict_keys(['weight', 'bias'])
权重类型: <class 'torch.nn.parameter.Parameter'>
权重数值 (Tensor):
tensor([[-0.0142,  0.1374,  0.0908, -0.1526,  0.2717, -0.0978, -0.0806,  0.0403]])
权重梯度: None


#### 1.2 一次性访问所有参数

我们常用 named_parameters() 来遍历整个网络的参数名和数值。

In [11]:
def print_all_params(net: nn.Module) -> None:
    """遍历并打印网络中所有参数的名称和形状。"""
    for name, param in net.named_parameters():
        print(f"参数名: {name:20} | 形状: {list(param.shape)}")

# 演示
print_all_params(net)

参数名: 0.weight             | 形状: [8, 4]
参数名: 0.bias               | 形状: [8]
参数名: 2.weight             | 形状: [1, 8]
参数名: 2.bias               | 形状: [1]


#### 1.3 从嵌套块搜集参数

In [12]:
# 生成块的函数
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())

def block2():
    net = nn.Sequential()
    for i in range(4):
        # 在此处嵌套
        net.add_module(f"block {i}", block1())
    return net

# --- 示例 ---
nest_net = nn.Sequential(block2(), nn.Linear(4, 1))
X = torch.randn(2, 4)
nest_net(X)

print(nest_net)

Sequential(
  (0): Sequential(
    (block 0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 1): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (block 3): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)


In [13]:
# 访问第一个块的，第二个子块的第一层的偏置项 (0-based)
nest_net[0][1][0].bias

Parameter containing:
tensor([ 0.2203,  0.2943,  0.4953, -0.1088,  0.3055, -0.1241, -0.4215,  0.1861],
       requires_grad=True)

---

### 2. 参数初始化

PyTorch 默认会进行初始化，但为了数值稳定性（4.8节内容），我们通常需要自定义初始化策略。


#### 2.1 使用内置初始化器


In [14]:
def init_normal(m: nn.Module) -> None:
    """内置正态分布初始化策略。"""
    if isinstance(m, nn.Linear):
        nn.init.normal_(m.weight, mean=0, std=0.01)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

def init_constant(m: nn.Module) -> None:
    """内置常数初始化策略。"""
    if isinstance(m, nn.Linear):
        nn.init.constant_(m.weight, 1)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

# --- 使用方法 ---
net.apply(init_normal) # 递归应用
net.apply(init_constant)

Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

### 2.2 自定义初始化逻辑

如果教材要求的逻辑非常特殊（比如：$w \sim \mathcal{U}(-10, 10)$ 且绝对值大于 5 才保留），我们可以直接操作 `.data`。


In [15]:
def my_custom_init(m: nn.Module) -> None:
    """实现复杂的自定义初始化逻辑。"""
    if isinstance(m, nn.Linear):
        print(f"正在初始化层: {m}")
        # 1. 均匀分布
        nn.init.uniform_(m.weight, -10, 10)
        # 2. 复杂的逻辑过滤：只保留绝对值 >= 5 的元素，其余置 0
        m.weight.data *= (m.weight.data.abs() >= 5)

# --- 示例 ---
net.apply(my_custom_init)

正在初始化层: Linear(in_features=4, out_features=8, bias=True)
正在初始化层: Linear(in_features=8, out_features=1, bias=True)


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

##### **注意**：

1. `.data` 是 PyTorch 的历史遗留属性，即使在初始化时使用 `.data` 是“安全”的，但它存在 **“静默错误”**：

    如果你习惯了使用 `.data`，万一在 `forward` 过程中误用了它（例如 `x.data += 1`），PyTorch 的自动求导系统**完全不会报错**，但最终算出来的梯度会是错的。这种 Bug 极难排查。


2. 初始化参数的现代标准做法：

    虽然 `.data` 在初始化时有效，但现在 PyTorch 官方建议使用 `torch.no_grad()`，因为它更显式且安全：

In [16]:
# 现代化的 PyTorch 自定义初始化
@torch.no_grad
def my_custom_init(m: nn.Module) -> None:
    """实现复杂的自定义初始化逻辑。"""
    if isinstance(m, nn.Linear):
        print(f"正在初始化层: {m}")
        # 1. 均匀分布
        nn.init.uniform_(m.weight, -10, 10)
        # 2. 复杂的逻辑过滤：只保留绝对值 >= 5 的元素，其余置 0
        # 直接对 Tensor 进行原位操作，不需要访问 .data
        m.weight *= (m.weight.abs() >= 5)

# --- 示例 ---
net.apply(my_custom_init)

正在初始化层: Linear(in_features=4, out_features=8, bias=True)
正在初始化层: Linear(in_features=8, out_features=1, bias=True)


Sequential(
  (0): Linear(in_features=4, out_features=8, bias=True)
  (1): ReLU()
  (2): Linear(in_features=8, out_features=1, bias=True)
)

---

### 3. 参数绑定与共享

有时我们希望在模型的不同层之间共享同一组权重。这在循环神经网络（RNN）或自编码器（Autoencoder）中非常常见。


#### 3.1 代码实现：同一个实例多次调用

实现共享参数最简单的方法是：先定义一个层实例，然后在 `forward` 中多次调用它。

> 实际上 5.1 中有演示过共享参数的情况。


In [17]:
class SharedParamMLP(nn.Module):
    """展示如何在不同层之间共享参数。"""
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        # 定义一个共享层
        self.shared = nn.Linear(hidden_dim, hidden_dim)

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            self.shared, # 第一次使用
            nn.ReLU(),
            self.shared, # 第二次使用（参数与上面完全绑定）
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, X: Tensor) -> Tensor:
        return self.net(X)
    
# --- 验证绑定 ---
net_shared = SharedParamMLP(4, 8, 1)
# 检查 Layer 2 和 Layer 4 的参数是否指向同一个内存地址
print(f"参数是否物理共享: {net_shared.net[2].weight is net_shared.net[4].weight}")

参数是否物理共享: True


**注意**: 由于它们指向同一个内存地址，在反向传播时，这个共享层的梯度是两次调用产生的梯度之和。

---

### 4. 细节补充

关于 `.data` 与 `.detach()` 的区别及用途：

* `.data` (不推荐)

    *   **行为**：它是指向 Tensor 数据的原始指针。
    *   **风险**：修改 `.data` 是**不安全**的。如果你在计算图中修改了它，PyTorch 的自动求导系统（Autograd）无法察觉，这会导致梯度计算错误而不报错。
    *   **用途**：主要用于兼容旧代码。在初始化参数时（如 `param.data.fill_(0)`），因为它不涉及梯度，所以看起来可行，但现代 PyTorch 更推荐在 `torch.no_grad()` 上下文中操作。

* `.detach()` (推荐)

    *   **行为**：返回一个新的 Tensor，从当前计算图中分离出来，但**共享相同的存储空间**。
    *   **安全机制**：它是安全的。如果你修改了 `detach()` 后的 Tensor，而原 Tensor 在反向传播中还需要用到，PyTorch 会报错（计算原 Tensor 的 In-place 修改错误），而不是产生错误的梯度。
    *   **典型用途**：
        *   **截断梯度传播**：在训练 GAN 时固定生成器更新判别器。
        *   **迁移学习**：冻结某些层的权重。
        *   **数值评估**：将输出转为 NumPy (通常用法是 `y.detach().cpu().numpy()`)。

* 总结对比

    | 特性 | `.data` | `.detach()` |
    | :--- | :--- | :--- |
    | **Autograd 监控** | 完全无视，修改可能导致静默的梯度错误 | 监控 In-place 修改，确保梯度计算安全 |
    | **共享内存** | 是 | 是 |
    | **推荐程度** | 不推荐 (Deprecated) | **推荐** |


In [18]:
# 参数提取练习
import torch
from torch import nn

# 创建 3 层 MLP 
net_test1 = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

# 提取并修改第一层偏置为 42.0
with torch.no_grad():
    net_test1[0].bias.fill_(42.0)

# 验证
print(f"第一层偏置: {net_test1[0].bias}")
print(f"验证结果: {'成功' if net_test1[0].bias[0] == 42.0 else '失败'}")

第一层偏置: Parameter containing:
tensor([42., 42., 42., 42., 42., 42., 42., 42.], requires_grad=True)
验证结果: 成功


In [19]:
# 共享参数练习

# 实例化共享参数模型
net_shared = SharedParamMLP(4, 8, 1)

# 模拟一轮训练
X = torch.randn(2, 4)
y = torch.randn(2, 1)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(net_shared.parameters(), lr=0.1)

# 开始训练
net_shared.train()
output = net_shared(X)
loss = loss_fn(output, y)
optimizer.zero_grad()
loss.backward()
optimizer.step()

# 验证 Layer 2 和 Layer 4 的参数是否仍指向同一个内存地址
weight2 = net_shared.net[2].weight
weight4 = net_shared.net[4].weight

print(f"物理地址是否相同 (is): {weight2 is weight4}")

# 细节补充：查看它们的梯度，共享层的梯度会是两次调用梯度的累加
print(f"共享层权重的梯度是否存在: {weight2.grad is not None}")

物理地址是否相同 (is): True
共享层权重的梯度是否存在: True


---


## 5.3 延后初始化

> 延后初始化允许我们只定义输出维度，让框架在第一次看到数据时自动推断输入维度。


### 1. 核心动机：为什么需要“懒加载”？

1.  **简化架构设计**：不需要手动计算全连接层的输入维度（特别是接在卷积层后面时）。
2.  **灵活性**：同一个模型定义可以适配不同尺寸的输入（只要逻辑允许）。
3.  **解耦**：模型定义不再强绑定于特定的输入形状。

---

### 2. PyTorch 的实现：`nn.Lazy` 模块

传统的 `nn.Linear` 在定义时就会分配内存并初始化权重。
PyTorch 引入了 **`nn.LazyLinear`**、**`nn.LazyConv2d`** 等模块来实现延后初始化。

In [ ]:
import torch
from torch import nn, Tensor

def deferred_init_demo() -> None:
    """演示 PyTorch 中的延迟初始化 (Lazy Initialization)。"""

    # 推荐实践：仅在 输入维度动态 或 计算复杂的层 使用 Lazy 
    # 后续层建议显式声明维度，以增强代码的可读性和严谨性

    # 1. 定义一个包含 Lazy 层的模型
    # 注意：我们没有指定输入维度 (in_features)
    net = nn.Sequential(
        nn.LazyLinear(256), # 第一层 Lazy：自动适配 X 的特征数
        nn.ReLU(),
        nn.Linear(256, 10) # 后续层显式：此处已知输入必为 256
    )

    # 2. 检查初始化前的状态
    # 此时参数尚未分配内存，打印出来会看到 <uninitialized>
    print(">>> 运行前向传播前的第一层权重: ")
    print(net[0].weight)

    # 3. 模拟输入数据
    X: Tensor = torch.randn(2, 20)

    # 4. 进行第一次前向传播 (触发初始化)
    net(X)

    # 5. 检查初始化后的状态
    print("\n>>> 运行前向传播后的第一层权重形状: ")
    # 框架自动推断出输入是 20，所以形状变为 [256, 20]
    # 权重矩阵 A 的形状是强制转置存储的，即 [in, out]
    print(net[0].weight.shape)
    print(f"权重数值已就绪: {net[0].weight.data.mean():.4f}")

deferred_init_demo()

>>> 运行前向传播前的第一层权重: 
<UninitializedParameter>

>>> 运行前向传播后的第一层权重形状: 
torch.Size([256, 20])
权重数值已就绪: -0.0007


---

### 3. 延后初始化的底层逻辑

当你调用 `net(X)` 时，PyTorch 内部发生了以下动作：

1.  **形状推断**：查看输入 `X` 的最后一个维度（即 20）。
2.  **内存分配**：根据推断出的形状 `(256, 20)` 立即创建权重张量。
3.  **参数初始化**：执行默认的初始化算法（或你通过 `net.apply` 指定的算法）。
4.  **计算流执行**：完成正常的矩阵乘法。

**注意**：一旦初始化完成，该层的形状就固定了。如果第二次传入一个维度不同的 `X`（比如输入维度变成 30），程序会报错。

---

### 4. 补充

#### 4.1 什么时候使用 Lazy 模式？
*   **推荐场景**：当你不想花时间去算 `Flatten` 之后的维度是多少时。
*   **不推荐场景**：如果你需要对模型进行非常精细的**参数初始化控制**（某些初始化算法可能依赖于明确的形状信息），延后初始化可能会让代码逻辑变得隐晦。

#### 4.2 初始化顺序的陷阱
如果你想自定义初始化逻辑，必须在**第一次前向传播之后**进行，或者使用特殊的钩子。

```python
# 错误做法：此时 net[0].weight 还是 uninitialized，调用 apply 会失效或报错
# net.apply(init_weights) 

# 正确做法：
net(X) # 先喂一次数据
net.apply(init_weights) # 再初始化
```

#### 4.3 显存管理
由于参数是延后创建的，这意味着你的显存占用会在训练的第一轮（第一个 Batch）突然跳升。在工程监控中，要留意这个瞬时增长。

---
